# Projet Creation un Chatbot : Assistant de restaurant






Nosaiba Elkrekshi

Jacques Callewaert

M1 Data & IA

In [36]:
!pip install -q streamlit pyngrok scikit-learn

Etape 1


# Partie 1: INSTALLATION DES DÉPENDANCES




In [37]:
import pandas as pd
import numpy as np
import re
import string
import pickle
import warnings
from datetime import datetime

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings('ignore')



In [38]:

# Télécharger ressources NLTK
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except:
        nltk.download(resource)

print(" Installation terminée\n")


 Installation terminée



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


# Partie 2: CHARGEMENT ET FUSION DES DONNÉES

In [39]:

def load_and_prepare_restaurants():
    """Charge et fusionne toutes les données des restaurants."""

    print("\n" + "="*70)
    print("ÉTAPE 1: PRÉPARATION DES DONNÉES RESTAURANTS")
    print("="*70 + "\n")

    try:
        # Chargement
        print("📂 Chargement des fichiers CSV...")
        places = pd.read_csv('/content/geoplaces2.csv', encoding='utf-8')
        ratings = pd.read_csv('/content/rating_final.csv', encoding='utf-8')
        cuisine = pd.read_csv('/content/chefmozcuisine.csv', encoding='utf-8')
        parking = pd.read_csv('/content/chefmozparking.csv', encoding='utf-8')
        payment = pd.read_csv('/content/chefmozaccepts.csv', encoding='utf-8')

        print(f"    {len(places)} restaurants")
        print(f"    {len(ratings)} avis")
        print(f"    {len(cuisine)} types de cuisine")

        # Fusion des données
        print("\n Fusion des données...")

        # Cuisines (plusieurs par restaurant)
        cuisine_grouped = cuisine.groupby('placeID')['Rcuisine'].apply(
            lambda x: ', '.join(x.unique())
        ).reset_index()
        cuisine_grouped.columns = ['placeID', 'cuisines']
        restaurants = places.merge(cuisine_grouped, on='placeID', how='left')

        # Paiements
        payment_grouped = payment.groupby('placeID')['Rpayment'].apply(
            lambda x: ', '.join(x.unique())
        ).reset_index()
        payment_grouped.columns = ['placeID', 'payment_methods']
        restaurants = restaurants.merge(payment_grouped, on='placeID', how='left')

        # Parkings
        parking_grouped = parking.groupby('placeID')['parking_lot'].apply(
            lambda x: ', '.join(x.unique())
        ).reset_index()
        parking_grouped.columns = ['placeID', 'parking_options']
        restaurants = restaurants.merge(parking_grouped, on='placeID', how='left')

        # Notes moyennes
        ratings_stats = ratings.groupby('placeID').agg({
            'rating': ['mean', 'count'],
            'food_rating': 'mean',
            'service_rating': 'mean'
        }).reset_index()
        ratings_stats.columns = ['placeID', 'avg_rating', 'nb_reviews',
                                 'avg_food_rating', 'avg_service_rating']
        restaurants = restaurants.merge(ratings_stats, on='placeID', how='left')

        # Nettoyage
        restaurants['avg_rating'] = restaurants['avg_rating'].fillna(0)
        restaurants['nb_reviews'] = restaurants['nb_reviews'].fillna(0)

        # Sauvegarde
        restaurants.to_csv('restaurants_prepared.csv', index=False)

        print(f"   ✅ Base créée: {len(restaurants)} restaurants")
        print(f"   ✅ Sauvegardé: 'restaurants_prepared.csv'")

        print("\n📊 Statistiques:")
        print(f"   - Villes: {restaurants['city'].nunique()}")
        print(f"   - Cuisines: {restaurants['cuisines'].dropna().str.split(', ').explode().nunique()}")
        print(f"   - Avec avis: {len(restaurants[restaurants['nb_reviews'] > 0])}")

        return restaurants, ratings

    except Exception as e:
        print(f"\n❌ Erreur: {e}")
        print("⚠️ Vérifiez que les fichiers CSV sont dans /content/")
        return None, None


# Exécution
restaurants_df, ratings_df = load_and_prepare_restaurants()

if restaurants_df is not None:
    print("\n✅ ÉTAPE 1 TERMINÉE - On passe à l'etape'2 \n")
else:
    print("\n❌ ERREUR - Corrigez avant de continuer\n")


ÉTAPE 1: PRÉPARATION DES DONNÉES RESTAURANTS

📂 Chargement des fichiers CSV...
    130 restaurants
    1161 avis
    916 types de cuisine

 Fusion des données...
   ✅ Base créée: 130 restaurants
   ✅ Sauvegardé: 'restaurants_prepared.csv'

📊 Statistiques:
   - Villes: 17
   - Cuisines: 23
   - Avec avis: 130

✅ ÉTAPE 1 TERMINÉE - On passe à l'etape'2 




ÉTAPE 2: CRÉATION DU CORPUS D'INTENTIONS ÉTENDU

In [40]:
def create_extended_corpus():
    """
    Crée un corpus étendu avec 600+ exemples pour améliorer les performances.
    Target: 70%+ d'accuracy (vs 41% avec petit corpus)
    """

    print("\n" + "="*70)
    print("ÉTAPE 2: CRÉATION DU CORPUS D'INTENTIONS ÉTENDU")
    print("="*70 + "\n")

    # Corpus étendu - 40+ exemples par intention minimum
    INTENTS = {
        'greeting': [
            # Formel (15)
            "bonjour", "bonsoir", "bon matin", "bonne journée", "bonne soirée",
            "salutations", "bien le bonjour", "bonjour à vous", "bonsoir à tous",
            "bonjour monsieur", "bonjour madame", "bien le bonsoir",
            "salutations distinguées", "mes salutations", "bonjour cher assistant",

            # Informel (20)
            "salut", "coucou", "hello", "hi", "hey", "yo", "wesh", "cc", "slt", "bjr",
            "ciao", "kikou", "yop", "heyo", "holà", "ola", "hello hello", "yo yo",
            "salut salut", "coucou toi",

            # Avec contexte (20)
            "bonjour comment vas-tu", "salut ça va", "hello comment ça va",
            "bonjour je cherche un restaurant", "hello je voudrais manger",
            "bonsoir j'ai faim", "salut j'aimerais trouver un resto",
            "coucou tu peux m'aider", "hey je cherche où manger",
            "bonjour j'ai besoin d'aide", "salut je veux un restaurant",
            "bonjour tu vas bien", "hey quoi de neuf", "salut tout va bien",
            "bonjour je peux te poser une question", "salut besoin d'aide",
            "hello j'ai une question", "yo c'est pour un resto", "salut dispo",
            "bonjour ça roule",
        ],

        'provide_location': [
            # Villes (20)
            "je suis à Lyon", "dans Paris", "à Marseille", "San Luis Potosi",
            "Cuernavaca", "victoria", "je suis sur Lyon", "à Paris",
            "dans Marseille", "san luis", "ciudad victoria", "slp",
            "dans la ville de Lyon", "ville de Paris", "Marseille centre",
            "San Luis Potosi Mexique", "je me trouve à Lyon", "situé à Paris",
            "localisé à Marseille", "je suis en vacances à Paris",

            # Quartiers/zones (20)
            "près de la Part-Dieu", "au Vieux Lyon", "à Bellecour",
            "dans le 2ème arrondissement", "dans le centre", "centre-ville",
            "quartier Part-Dieu", "secteur Bellecour", "zone centre",
            "dans le centre de Lyon", "au centre ville", "downtown",
            "près du centre", "aux alentours du centre", "secteur centre",
            "quartier latin", "dans le vieux", "vieille ville",
            "près de la gare", "coin gare",

            # Vague (15)
            "dans ma ville", "ici", "près de chez moi", "dans mon quartier",
            "à côté", "pas loin", "dans le coin", "aux alentours",
            "dans les parages", "coin sympa", "par ici", "près d'ici",
            "dans le secteur", "dans les environs", "coin du centre",

            # Avec verbe (15)
            "je cherche à Lyon", "vers Lyon", "je veux manger à Paris",
            "cherche restaurant Lyon", "restaurant sur Paris", "manger à Marseille",
            "je visite Lyon", "je travaille à Paris", "je suis en balade à Lyon",
            "je passe par Paris", "je reste à Lyon", "je suis de passage à Paris",
            "je me balade à Lyon", "je traine à Paris", "je squatte Lyon",
        ],

        'provide_cuisine': [
            # Principales (15)
            "je veux manger italien", "j'aime le mexicain", "cuisine japonaise",
            "un restaurant français", "de la nourriture chinoise", "américain",
            "italien", "mexicain", "japonais", "français", "chinois",
            "cuisine italienne", "resto mexicain", "food japonais", "style américain",

            # Plats spécifiques (20)
            "des pizzas", "des sushis", "tacos", "burger", "des pâtes",
            "ramen", "burritos", "enchiladas", "sashimi", "tempura",
            "pad thai", "pho", "nems", "spring rolls", "makis",
            "california rolls", "carbonara", "margherita", "pepperoni", "hawaii",

            # Préférences (15)
            "je préfère l'italien", "j'adore la cuisine mexicaine",
            "j'aimerais manger japonais", "je raffole de l'italien",
            "fan de cuisine mexicaine", "amateur de sushi", "accro à la pizza",
            "plutôt mexicain", "pas mal le japonais", "tenter le mexicain",
            "essayer le japonais", "j'aime bien l'italien", "pourquoi pas italien",
            "ça me dit mexicain", "envie d'italien",

            # Envies (15)
            "envie de pizza", "j'ai envie de sushis", "une petite envie d'italien",
            "ça me dit du mexicain", "je me ferais bien un burger",
            "un bon ramen", "petite faim de pizza", "j'ai la dalle de sushi",
            "je crève de faim italien", "trop envie de mexicain",
            "mort de faim japonais", "besoin d'une pizza", "urgent burger",
            "vite un taco", "immédiatement des pâtes",

            # Généraux (15)
            "fast food", "restauration rapide", "street food", "bar", "pub",
            "brasserie", "bistrot", "cafe", "cafeteria", "coffee shop",
            "family restaurant", "restaurant familial", "gastronomique",
            "cuisine du monde", "international",
        ],

        'list_restaurants': [
            # Direct (15)
            "quels restaurants disponibles", "montre-moi les restaurants",
            "liste les restaurants", "affiche les restaurants",
            "tous les restaurants", "liste complète", "liste-moi les restos",
            "fais-moi voir les restaurants", "balance les restaurants",
            "envoie la liste", "file les restos", "donne la liste complète",
            "affiche tout", "montre tout", "liste tout",

            # Questions (15)
            "qu'est-ce que tu as comme restaurants", "quels restos tu as",
            "quels sont les restaurants", "que proposes-tu comme restaurants",
            "tu connais quoi comme restos", "c'est quoi les options",
            "quelles sont mes options", "qu'est-ce qu'il y a",
            "t'as quoi", "tu proposes quoi", "options disponibles",
            "choix possibles", "alternatives", "suggestions", "propositions",

            # Poli (15)
            "donne-moi des restaurants", "propose-moi des restaurants",
            "suggère-moi des restaurants", "recommande-moi des restos",
            "trouve-moi des restaurants", "cherche des restaurants",
            "tu pourrais me montrer", "serais-tu me proposer",
            "peux-tu me suggérer", "pourrais-je voir", "puis-je avoir la liste",
            "aurais-tu des suggestions", "des idées de restos",
            "conseil de restaurant", "aide-moi à choisir",

            # Général (15)
            "où puis-je manger", "où manger", "où est-ce qu'on peut manger",
            "j'aimerais voir les restaurants", "montre les options",
            "qu'est-ce que tu proposes", "inspire moi", "conseille moi",
            "propose moi quelque chose", "trouve moi un truc", "cherche moi un resto",
            "donne moi des idées", "aide-moi", "guide-moi", "oriente-moi",
        ],

        'get_reviews': [
            # Avis (15)
            "quels sont les avis", "montre-moi les avis", "les avis clients",
            "avis clients", "donne-moi les avis", "j'aimerais voir les avis",
            "voir les avis", "consulter les avis", "lire les avis",
            "check les avis", "regarde les avis", "montre avis",
            "affiche avis", "balance les avis", "file les avis",

            # Commentaires (10)
            "les commentaires", "commentaires clients", "qu'est-ce qu'on dit",
            "que disent les gens", "qu'en pensent les clients", "opinions",
            "retours clients", "feedback", "témoignages", "expériences",

            # Notes (15)
            "les notes", "quelle note", "évaluation", "c'est bien noté",
            "bien évalué", "bonne note", "mauvaise note", "note moyenne",
            "rating", "score", "classement", "notation", "appréciation",
            "cote", "popularité",

            # Réputation (15)
            "bonne réputation", "quelle réputation", "c'est réputé",
            "connu ou pas", "les retours", "retours d'expérience",
            "c'est bon", "c'est bien", "c'est recommandé", "les gens aiment",
            "apprécié", "qu'est-ce qu'ils disent", "critiques", "c'est populaire",
            "c'est tendance",
        ],

        'get_address': [
            # Direct (15)
            "quelle est l'adresse", "l'adresse", "adresse svp", "adresse stp",
            "donne-moi l'adresse", "indique-moi l'adresse",
            "j'aimerais l'adresse", "je veux l'adresse", "balance l'adresse",
            "file l'adresse", "montre l'adresse", "envoie l'adresse",
            "donne adresse", "adresse du resto", "adresse exacte",

            # Localisation (15)
            "où se trouve ce restaurant", "où est-il", "où c'est",
            "c'est où", "il est où", "localisation", "position", "emplacement",
            "situé où", "se situe où", "on le trouve où", "localisé où",
            "géolocalisation", "lieu exact", "endroit précis",

            # Navigation (15)
            "comment y aller", "comment aller", "itinéraire", "trajet", "chemin",
            "pour y aller", "comment se rendre", "route", "parcours",
            "direction", "comment j'y vais", "pour aller là-bas",
            "GPS", "coordonnées GPS", "latitude longitude",

            # Questions (15)
            "à quelle adresse", "dans quelle rue", "quel numéro", "quelle rue",
            "numéro de rue", "nom de la rue", "c'est quelle adresse",
            "dis-moi où c'est", "tu peux me dire où", "montre-moi où",
            "indique où", "précise l'adresse", "détails adresse",
            "adresse complète", "adresse postale",
        ],

        'get_menu': [
            # Menu (15)
            "quel est le menu", "le menu", "menu svp", "menu stp",
            "montre-moi le menu", "j'aimerais voir le menu",
            "affiche le menu", "donne le menu", "envoie le menu",
            "balance le menu", "file le menu", "voir le menu",
            "consulter le menu", "regarder le menu", "check le menu",

            # Carte (10)
            "la carte", "carte du restaurant", "montre la carte", "voir la carte",
            "affiche la carte", "donne la carte", "consulte carte",
            "regarde carte", "carte des plats", "menu carte",

            # Plats (20)
            "les plats", "quels plats", "liste des plats", "plats disponibles",
            "qu'est-ce qu'on peut manger", "quels plats proposent-ils",
            "plats au menu", "choix de plats", "sélection plats",
            "variété plats", "qu'est-ce qu'ils servent", "que proposent-ils",
            "qu'est-ce qu'il y a à manger", "nourriture proposée",
            "options de menu", "choix disponibles", "qu'est-ce qu'ils font",
            "leurs plats", "plats du jour", "suggestions plats",

            # Spécialités (15)
            "spécialités", "quelles sont les spécialités", "plats signature",
            "spécialité de la maison", "plat du jour", "suggestions",
            "spéciaux", "recommandations", "must-try", "incontournables",
            "best-of", "top plats", "plats phares", "vedettes", "stars menu",
        ],

        'get_prices': [
            # Prix (20)
            "quel est le prix", "les prix", "prix svp", "prix stp",
            "combien ça coûte", "combien", "coût", "quel prix",
            "c'est combien", "ça coûte combien", "montant", "somme",
            "tarif", "tarifs", "tarification", "les tarifs",
            "grille tarifaire", "barème", "facturation", "coût total",

            # Budget (20)
            "c'est cher", "pas cher", "cher ou pas", "gamme de prix",
            "fourchette de prix", "budget", "dans quel budget",
            "niveau prix", "ordre de prix", "économique", "abordable",
            "bon marché", "rapport qualité prix", "prix raisonnables",
            "accessible", "raisonnable", "modéré", "élevé", "bas prix", "haut prix",

            # Questions (20)
            "je dois payer combien", "quel est le tarif", "prix moyen",
            "prix des plats", "combien coûte un plat", "prix du menu",
            "combien pour deux", "prix pour une personne", "prix par tête",
            "combien pour 4", "addition moyenne", "note moyenne",
            "coût repas", "budget repas", "dépense moyenne",
            "fourchette budget", "estimation prix", "idée du prix",
            "ordre de grandeur", "prix approximatif",
        ],

        'get_hours': [
            # Horaires (15)
            "horaires d'ouverture", "horaires", "heures d'ouverture",
            "ouvert quand", "c'est ouvert quand", "à quelle heure",
            "jusqu'à quelle heure", "planning horaire", "emploi du temps",
            "quand puis-je venir", "disponibilités", "créneaux horaires",
            "plages horaires", "heures", "timing",

            # Ouverture/Fermeture (15)
            "ils ouvrent quand", "heure d'ouverture", "ils ferment à quelle heure",
            "heure de fermeture", "ferme quand", "ouvre à quelle heure",
            "ouverture", "fermeture", "début service", "fin service",
            "ouvrent à quelle heure", "ferment quand", "horaire ouverture",
            "horaire fermeture", "heure début",

            # État (15)
            "c'est ouvert maintenant", "ouvert maintenant", "fermé ou ouvert",
            "encore ouvert", "ouvert aujourd'hui", "fermé aujourd'hui",
            "actuellement ouvert", "disponible maintenant", "accessible maintenant",
            "on peut y aller maintenant", "c'est fermé", "c'est ouvert là",
            "ouvert en ce moment", "fermé en ce moment", "dispo maintenant",

            # Jours (15)
            "ouvert le dimanche", "ouvert le soir", "ouvert le midi",
            "service continu", "horaires du restaurant", "jours d'ouverture",
            "quels jours", "ouvert tous les jours", "ouvert weekend",
            "ouvert semaine", "fermé dimanche", "fermé lundi", "7j/7",
            "planning restaurant", "calendrier ouverture",
        ],

        'make_reservation': [
            # Direct (15)
            "je veux réserver", "réservation", "réserver", "puis-je réserver",
            "est-ce que je peux réserver", "faire une réservation",
            "prendre une réservation", "j'aimerais réserver",
            "besoin de réserver", "faut réserver", "comment réserver",
            "réservation possible", "peut-on réserver", "réserver svp", "resa",

            # Table (15)
            "réserver une table", "table pour deux", "table pour 4 personnes",
            "table de 6", "booker une table", "réserver une place",
            "table pour 2", "table 4 couverts", "table 6 personnes",
            "réserver places", "booking table", "reserve table",
            "table disponible", "place disponible", "couverts disponibles",

            # Quand (15)
            "réserver ce soir", "réservation pour demain", "réserver pour samedi",
            "table pour ce weekend", "réserver maintenant", "réservation immédiate",
            "ce soir", "demain soir", "samedi midi", "dimanche soir",
            "pour ce weekend", "pour vendredi", "prochaine semaine",
            "dans 2 jours", "pour 19h",

            # Disponibilité (15)
            "disponibilité", "places disponibles", "encore de la place",
            "complet ou pas", "il reste des tables", "possible de réserver",
            "vous avez de la place", "y'a encore des tables", "c'est plein",
            "dispo", "disponible", "libre", "occupé", "complet", "places libres",
        ],

        'get_parking': [
            # Général (15)
            "y a-t-il un parking", "parking", "stationnement",
            "où se garer", "où garer", "place de parking",
            "parking disponible", "il y a un parking", "est-ce qu'il y a un parking",
            "où puis-je me garer", "où stationner", "pour se garer",
            "garer où", "stationnement où", "parking possible",

            # Types (15)
            "parking payant", "parking gratuit", "parking privé", "parking public",
            "garage", "place de stationnement", "parking souterrain",
            "parking extérieur", "parking couvert", "box", "emplacement",
            "place voiture", "parking véhicule", "park auto", "parc auto",

            # Questions (10)
            "comment se garer", "garer la voiture où", "stationnement facile",
            "problème de parking", "parking proche", "parking à côté",
            "parking près", "facile de se garer", "stationnement aisé",
            "place facilement",
        ],

        'get_payment': [
            # Général (10)
            "moyens de paiement", "paiement", "comment payer",
            "modes de paiement", "paiements acceptés", "moyens acceptés",
            "options paiement", "façons de payer", "modalités paiement",
            "types paiement",

            # Cartes (15)
            "ils acceptent les cartes", "carte bancaire", "carte bancaire acceptée",
            "payer par carte", "je peux payer par carte", "CB acceptée",
            "visa mastercard", "cartes de crédit", "carte bleue",
            "paiement carte", "acceptent CB", "carte ok", "CB possible",
            "cartes oui", "accepte carte",

            # Espèces (10)
            "espèces", "cash", "liquide", "payer en espèces",
            "acceptent le liquide", "cash accepté", "paiement liquide",
            "espèces ok", "liquide possible", "cash possible",

            # Autres (15)
            "chèques", "tickets restaurant", "paiement mobile", "Apple Pay",
            "sans contact", "contactless", "NFC", "Google Pay",
            "Samsung Pay", "paiement électronique", "virement",
            "paypal", "wallet", "portefeuille électronique", "crypto",
        ],

        'thanks': [
            # Simple (20)
            "merci", "merci beaucoup", "merci bien", "grand merci",
            "mille mercis", "merci infiniment", "merci mille fois",
            "merci énormément", "merci bcp", "mrc", "thx", "thanks",
            "thank you", "ty", "cheers", "merci à toi", "merci à vous",
            "merci du fond du coeur", "un grand merci", "merci vraiment",

            # Avec contexte (20)
            "super merci", "parfait merci", "ok merci", "d'accord merci",
            "merci pour l'info", "merci de ton aide", "merci pour ton aide",
            "génial merci", "cool merci", "top merci", "nickel merci",
            "impeccable merci", "excellent merci", "formidable merci",
            "merci c'est parfait", "merci beaucoup pour tout",
            "merci c'est gentil", "merci c'est sympa", "merci t'es cool",
            "merci t'assures",
        ],

        'goodbye': [
            # Formel (15)
            "au revoir", "à bientôt", "à plus tard", "bonne journée",
            "bonne soirée", "bonne continuation", "à la prochaine",
            "au plaisir", "à une prochaine fois", "à la revoyure",
            "adieu", "portez-vous bien", "prenez soin de vous",
            "à très bientôt", "à très vite",

            # Informel (20)
            "bye", "à plus", "à+", "salut", "ciao", "tchao", "bisous", "bises",
            "biz", "kiss", "peace", "cya", "see ya", "bb", "à++",
            "aller salut", "bon ben salut", "ok bye", "bon allez",
            "allez bye",

            # Départ (15)
            "je dois y aller", "je me sauve", "je file", "je pars", "j'y vais",
            "faut que j'y aille", "je te laisse", "je dois filer",
            "il faut que je parte", "je dois partir", "on se quitte",
            "je me casse", "je me tire", "je décarre", "je dégage",
        ],

        'unknown': [
            # Temps (10)
            "quelle heure est-il", "il est quelle heure", "l'heure",
            "il est quelle heure", "donne-moi l'heure", "heure actuelle",
            "heure qu'il est", "quelle heure", "il est combien", "combien d'heure",

            # Météo (10)
            "quel temps fait-il", "météo", "il fait beau", "va pleuvoir",
            "il pleut", "soleil", "nuages", "température", "climat", "temps qu'il fait",

            # Identité (10)
            "tu t'appelles comment", "qui es-tu", "c'est quoi ton nom", "tu es qui",
            "ton nom", "ton identité", "tu es quoi", "ton prénom", "comment tu t'appelles",
            "quel est ton nom",

            # État (10)
            "comment ça va", "ça va bien", "tu vas bien", "quoi de neuf",
            "comment tu vas", "ça roule", "tout va bien", "comment allez-vous",
            "vous allez bien", "ça va ou quoi",

            # Hors sujet (20)
            "parle-moi de toi", "tu fais quoi", "raconte une blague",
            "dis-moi une blague", "actualités", "news", "infos", "sport", "foot",
            "politique", "film", "série", "musique", "je m'ennuie", "amuse-moi",
            "chante", "danse", "joue", "calcule 2+2", "résous une équation",
        ]
    }

    # Créer DataFrame
    data = []
    for intent, examples in INTENTS.items():
        for example in examples:
            data.append({'text': example, 'intent': intent})

    df = pd.DataFrame(data)
    df.to_csv('training_corpus.csv', index=False)

    print(f"✅ Corpus créé: {len(df)} exemples")
    print(f"✅ {df['intent'].nunique()} intentions")
    print(f"\n📊 Répartition:")
    print(df['intent'].value_counts().to_string())
    print(f"\n✅ Sauvegardé: 'training_corpus.csv'")

    return df


# Exécution
corpus_df = create_extended_corpus()

print("\n✅ ÉTAPE 2 TERMINÉE - On passe à l'etape 3\n")


ÉTAPE 2: CRÉATION DU CORPUS D'INTENTIONS ÉTENDU

✅ Corpus créé: 860 exemples
✅ 15 intentions

📊 Répartition:
intent
provide_cuisine     80
provide_location    70
list_restaurants    60
get_menu            60
get_address         60
get_hours           60
get_prices          60
make_reservation    60
unknown             60
get_reviews         55
greeting            55
goodbye             50
get_payment         50
get_parking         40
thanks              40

✅ Sauvegardé: 'training_corpus.csv'

✅ ÉTAPE 2 TERMINÉE - On passe à l'etape 3



Etape 3 : PIPELINE NLP COMPLÈTE (PREPROCESSING + VECTORISATION + CLASSIFICATION)

In [41]:
# ============================================================================
# CLASSE 1: PREPROCESSING
# ============================================================================

class TextPreprocessor:
    """
    Preprocessing des textes (tokenisation, stop-words, lemmatisation).
    """

    def __init__(self, language='french'):
        self.language = language
        self.stop_words = set(stopwords.words(language))
        self.lemmatizer = WordNetLemmatizer()

    def clean_text(self, text):
        """Nettoyage: minuscules, ponctuation, espaces."""
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(self, text):
        """Tokenisation."""
        return word_tokenize(text, language=self.language)

    def remove_stopwords(self, tokens):
        """Suppression des stop-words."""
        return [token for token in tokens if token not in self.stop_words]

    def lemmatize(self, tokens):
        """Lemmatisation."""
        return [self.lemmatizer.lemmatize(token) for token in tokens]

    def preprocess(self, text):
        """Pipeline complet de preprocessing."""
        # 1. Nettoyage
        text = self.clean_text(text)

        # 2. Tokenisation
        tokens = self.tokenize(text)

        # 3. Stop-words
        tokens = self.remove_stopwords(tokens)

        # 4. Lemmatisation
        tokens = self.lemmatize(tokens)

        # 5. Reconstituer le texte
        return ' '.join(tokens)


# ============================================================================
# CLASSE 2: CLASSIFICATION
# ============================================================================

class IntentClassifier:
    """
    Classifier pour prédire les intentions.
    Compare plusieurs modèles et optimise le meilleur.
    """

    def __init__(self):
        self.models = {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'SVM': SVC(kernel='linear', probability=True, random_state=42),
        }

        self.best_model = None
        self.best_model_name = None
        self.best_score = 0

    def train_all_models(self, X_train, y_train, X_test, y_test):
        """Entraîne tous les modèles et compare."""

        results = {}

        print("\n🤖 Entraînement des modèles:")
        print("-" * 70)

        for name, model in self.models.items():
            print(f"\n⏳ {name}...")

            # Entraînement
            model.fit(X_train, y_train)

            # Prédictions
            y_pred = model.predict(X_test)

            # Métriques
            accuracy = accuracy_score(y_test, y_pred)

            # Cross-validation
            cv_scores = cross_val_score(model, X_train, y_train, cv=5)
            cv_mean = cv_scores.mean()

            results[name] = {
                'model': model,
                'accuracy': accuracy,
                'cv_score': cv_mean,
                'predictions': y_pred
            }

            print(f"    Accuracy Test: {accuracy:.4f}")
            print(f"    CV Score: {cv_mean:.4f}")

            # Garder le meilleur
            if accuracy > self.best_score:
                self.best_score = accuracy
                self.best_model = model
                self.best_model_name = name

        print("\n" + "="*70)
        print(f"🏆 MEILLEUR MODÈLE: {self.best_model_name}")
        print(f"   Accuracy: {self.best_score:.4f}")
        print("="*70)

        return results

    def optimize_hyperparameters(self, X_train, y_train):
        """Optimise les hyperparamètres du meilleur modèle."""

        print(f"\n🔧 Optimisation de {self.best_model_name}...")

        param_grids = {
            'Random Forest': {
                'n_estimators': [50, 100, 200],
                'max_depth': [10, 20, None],
                'min_samples_split': [2, 5]
            },
            'SVM': {
                'C': [0.1, 1, 10],
                'kernel': ['linear', 'rbf']
            },
            'Logistic Regression': {
                'C': [0.1, 1, 10],
                'penalty': ['l2']
            }
        }

        if self.best_model_name not in param_grids:
            print(f"   ⚠️ Pas de grid search pour {self.best_model_name}")
            return self.best_model

        grid_search = GridSearchCV(
            self.models[self.best_model_name],
            param_grids[self.best_model_name],
            cv=5,
            scoring='accuracy',
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)

        print(f"    Meilleurs params: {grid_search.best_params_}")
        print(f"    Meilleur score CV: {grid_search.best_score_:.4f}")

        self.best_model = grid_search.best_estimator_

        return grid_search.best_estimator_

    def predict(self, X):
        """Prédit l'intention."""
        if self.best_model is None:
            raise ValueError("Modèle pas entraîné!")
        return self.best_model.predict(X)

    def predict_proba(self, X):
        """Prédit les probabilités."""
        if self.best_model is None:
            raise ValueError("Modèle pas entraîné!")

        if hasattr(self.best_model, 'predict_proba'):
            return self.best_model.predict_proba(X)
        return None


# ============================================================================
# CLASSE 3: PIPELINE COMPLÈTE
# ============================================================================

class NLPPipeline:
    """
    Pipeline NLP complète intégrant preprocessing, vectorisation et classification.
    """

    def __init__(self, vectorization_method='tfidf', ngram_range=(1, 2)):
        self.preprocessor = TextPreprocessor(language='french')
        self.vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=1000
        )
        self.classifier = IntentClassifier()

    def fit(self, texts, labels, optimize=True):
        """Entraîne la pipeline complète."""

        print("\n" + "="*70)
        print("🚀 PIPELINE NLP - ENTRAÎNEMENT")
        print("="*70)

        # 1. Preprocessing
        print("\n1️⃣ Preprocessing des textes...")
        processed_texts = []
        for i, text in enumerate(texts):
            if i % 100 == 0:
                print(f"   Traitement: {i}/{len(texts)}", end='\r')
            processed_texts.append(self.preprocessor.preprocess(text))

        print(f"\n   {len(processed_texts)} textes prétraités")

        # 2. Vectorisation
        print("\n2️⃣ Vectorisation TF-IDF...")
        X = self.vectorizer.fit_transform(processed_texts)
        print(f"   ✓ Shape: {X.shape}")
        print(f"   ✓ Features: {X.shape[1]}")
        print(f"   ✓ N-grams: {self.vectorizer.ngram_range}")

        # 3. Split train/test
        print("\n3️⃣ Split train/test (80/20)...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, labels, test_size=0.2, random_state=42, stratify=labels
        )
        print(f"   Train: {X_train.shape[0]} exemples")
        print(f"   Test: {X_test.shape[0]} exemples")

        # 4. Classification
        print("\n4️⃣ Classification...")
        results = self.classifier.train_all_models(X_train, y_train, X_test, y_test)

        # 5. Optimisation
        if optimize:
            print("\n5️⃣ Optimisation des hyperparamètres...")
            self.classifier.optimize_hyperparameters(X_train, y_train)

            # Réévaluer
            y_pred_optimized = self.classifier.predict(X_test)
            accuracy_optimized = accuracy_score(y_test, y_pred_optimized)
            print(f"\n   ✓ Accuracy après optimisation: {accuracy_optimized:.4f}")

        # 6. Rapport final
        print("\n" + "="*70)
        print("📊 RAPPORT DE CLASSIFICATION FINAL")
        print("="*70 + "\n")

        y_pred_final = self.classifier.predict(X_test)

        print(classification_report(y_test, y_pred_final))

        # Matrice de confusion (top 5 erreurs)
        print("\n🔍 Analyse des erreurs (top 5):")
        cm = confusion_matrix(y_test, y_pred_final)

        # Trouver les plus grosses erreurs
        errors = []
        labels_unique = sorted(set(y_test))
        for i, true_label in enumerate(labels_unique):
            for j, pred_label in enumerate(labels_unique):
                if i != j and cm[i][j] > 0:
                    errors.append((cm[i][j], true_label, pred_label))

        errors.sort(reverse=True)
        for count, true, pred in errors[:5]:
            print(f"   • {count}x: '{true}' confondu avec '{pred}'")

        print("\n✅ Entraînement terminé !")

        return results

    def predict(self, text, return_confidence=True):
        """Prédit l'intention d'un texte."""

        # Preprocessing
        processed_text = self.preprocessor.preprocess(text)

        # Vectorisation
        X = self.vectorizer.transform([processed_text])

        # Prédiction
        intent = self.classifier.predict(X)[0]

        if return_confidence:
            probas = self.classifier.predict_proba(X)
            if probas is not None:
                confidence = np.max(probas[0])
                return intent, confidence
            else:
                return intent, 1.0

        return intent

    def save(self, filepath='chatbot_pipeline.pkl'):
        """Sauvegarde la pipeline."""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
        print(f"\n💾 Pipeline sauvegardée: '{filepath}'")

    @staticmethod
    def load(filepath='chatbot_pipeline.pkl'):
        """Charge une pipeline sauvegardée."""
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print(f"✓ Pipeline chargée: '{filepath}'")
        return pipeline


# ============================================================================
# EXÉCUTION
# ============================================================================

def train_nlp_pipeline():
    """Fonction principale pour entraîner la pipeline."""

    print("\n" + "="*70)
    print("ÉTAPE 3: ENTRAÎNEMENT DE LA PIPELINE NLP")
    print("="*70 + "\n")

    try:
        # Charger le corpus
        print("📂 Chargement du corpus...")
        corpus = pd.read_csv('training_corpus.csv', encoding='utf-8')
        print(f"   ✓ {len(corpus)} exemples chargés")
        print(f"   ✓ {corpus['intent'].nunique()} intentions")

        # Créer la pipeline
        print("\n🔧 Création de la pipeline NLP...")
        pipeline = NLPPipeline(
            vectorization_method='tfidf',
            ngram_range=(1, 2)  # Unigrams + Bigrams
        )

        # Entraîner
        pipeline.fit(
            texts=corpus['text'].tolist(),
            labels=corpus['intent'].tolist(),
            optimize=True  # Optimiser les hyperparamètres
        )

        # Sauvegarder
        pipeline.save('chatbot_pipeline.pkl')

        # Tests rapides
        print("\n" + "="*70)
        print(" TESTS RAPIDES")
        print("="*70 + "\n")

        test_phrases = [
            "bonjour comment vas-tu",
            "je cherche un restaurant à Lyon",
            "j'aime la cuisine italienne",
            "montre-moi les avis",
            "quelle est l'adresse",
            "au revoir"
        ]

        for phrase in test_phrases:
            intent, confidence = pipeline.predict(phrase)
            print(f"'{phrase}'")
            print(f"   → Intent: {intent} (Confiance: {confidence:.2f})\n")

        print("="*70)
        print("✅ ÉTAPE 3 TERMINÉE - On passe à la Partie 4")
        print("="*70 + "\n")

        return pipeline

    except FileNotFoundError:
        print("\n❌ ERREUR: Fichier 'training_corpus.csv' introuvable!")
        print("⚠️ Exécutez d'abord la Partie 2\n")
        return None

    except Exception as e:
        print(f"\n❌ ERREUR: {e}")
        import traceback
        traceback.print_exc()
        return None


# Exécution
pipeline = train_nlp_pipeline()

if pipeline is not None:
    print("\n🎉 Pipeline prête pour le chatbot !")
else:
    print("\n⚠️ Corrigez les erreurs avant de continuer")


ÉTAPE 3: ENTRAÎNEMENT DE LA PIPELINE NLP

📂 Chargement du corpus...
   ✓ 860 exemples chargés
   ✓ 15 intentions

🔧 Création de la pipeline NLP...

🚀 PIPELINE NLP - ENTRAÎNEMENT

1️⃣ Preprocessing des textes...
   Traitement: 800/860
   860 textes prétraités

2️⃣ Vectorisation TF-IDF...
   ✓ Shape: (860, 1000)
   ✓ Features: 1000
   ✓ N-grams: (1, 2)

3️⃣ Split train/test (80/20)...
   Train: 688 exemples
   Test: 172 exemples

4️⃣ Classification...

🤖 Entraînement des modèles:
----------------------------------------------------------------------

⏳ Logistic Regression...
    Accuracy Test: 0.6860
    CV Score: 0.6396

⏳ Random Forest...
    Accuracy Test: 0.6686
    CV Score: 0.6628

⏳ SVM...
    Accuracy Test: 0.6860
    CV Score: 0.6598

🏆 MEILLEUR MODÈLE: Logistic Regression
   Accuracy: 0.6860

5️⃣ Optimisation des hyperparamètres...

🔧 Optimisation de Logistic Regression...
    Meilleurs params: {'C': 10, 'penalty': 'l2'}
    Meilleur score CV: 0.6831

   ✓ Accuracy après optim

Etape 4 : INTERFACE STREAMLIT

In [42]:
# ============================================================================
# CLASSE CHATBOT COMPLÈTE
# ============================================================================

class RestaurantChatbot:
    """
    Chatbot d'assistance restaurant avec NLP et gestion de dialogue.
    """

    def __init__(self, restaurants_df, nlp_pipeline, confidence_threshold=0.70):
        self.restaurants = restaurants_df
        self.pipeline = nlp_pipeline
        self.confidence_threshold = confidence_threshold

        # Contexte conversationnel
        self.context = {
            'ville': None,
            'type_cuisine': None,
            'restaurant_courant': None,
            'restaurants_filtres': None,
            'conversation_started': False
        }

        # Historique
        self.history = []

        # Réponses variées
        self.responses = {
            'greeting': [
                "Bonjour ! 🍽️ Je suis votre assistant restaurant. Dans quelle ville cherchez-vous ?",
                "Salut ! Je peux vous aider à trouver un restaurant. Quelle ville ?",
                "Hello ! Bienvenue. Pour commencer, dans quelle ville ?",
                "Bonsoir ! Je suis là pour vous aider. Quelle ville vous intéresse ?"
            ],
            'thanks': [
                "De rien ! 😊 Besoin d'autre chose ?",
                "Avec plaisir ! N'hésitez pas.",
                "Je suis là pour ça !",
                "Content d'aider ! 👍"
            ],
            'goodbye': [
                "Au revoir ! Bon appétit ! 🍽️",
                "À bientôt ! Profitez bien !",
                "Bye ! N'hésitez pas à revenir !",
                "À plus ! Bonne journée !"
            ],
            'unknown': [
                "Désolé, je ne comprends pas. Reformulez ?",
                "Hmm, pas sûr de comprendre. Je vous aide avec les restaurants.",
                "Je ne suis pas sûr. Essayez de me demander sur les restaurants.",
                "Pardon ? Je suis spécialisé dans les restaurants."
            ]
        }

    # ========================================================================
    # EXTRACTION D'ENTITÉS
    # ========================================================================

    def extract_city(self, text):
        """Extrait la ville du texte."""
        text_lower = text.lower()

        # Villes de la base de données
        cities = self.restaurants['city'].dropna().unique()
        for city in cities:
            if city.lower() in text_lower:
                return city

        # Villes communes non dans la base
        common_cities = {
            'lyon': 'Lyon', 'paris': 'Paris', 'marseille': 'Marseille',
            'slp': 'San Luis Potosi', 'san luis': 'San Luis Potosi'
        }
        for key, value in common_cities.items():
            if key in text_lower:
                return value

        return None

    def extract_cuisine(self, text):
        """Extrait le type de cuisine."""
        text_lower = text.lower()

        # Types de la base
        if 'cuisines' in self.restaurants.columns:
            all_cuisines = self.restaurants['cuisines'].dropna().str.split(', ').explode()
            cuisines = all_cuisines.unique()
            for cuisine in cuisines:
                if cuisine.lower() in text_lower:
                    return cuisine

        # Mots-clés communs
        cuisine_keywords = {
            'Mexican': ['mexicain', 'mexican', 'tacos', 'burrito', 'enchilada'],
            'Italian': ['italien', 'italian', 'pizza', 'pâtes', 'pasta'],
            'Japanese': ['japonais', 'japanese', 'sushi', 'ramen', 'sashimi'],
            'French': ['français', 'french'],
            'Chinese': ['chinois', 'chinese'],
            'American': ['américain', 'american', 'burger'],
            'Fast_Food': ['fast food', 'rapide'],
            'Bar': ['bar', 'pub'],
            'Cafeteria': ['cafe', 'cafeteria']
        }

        for cuisine_type, keywords in cuisine_keywords.items():
            for keyword in keywords:
                if keyword in text_lower:
                    return cuisine_type

        return None

    def extract_restaurant_name(self, text):
        """Extrait le nom d'un restaurant."""
        text_lower = text.lower()

        # Chercher dans les restaurants filtrés d'abord
        if self.context['restaurants_filtres'] is not None:
            for _, restaurant in self.context['restaurants_filtres'].iterrows():
                if restaurant['name'].lower() in text_lower:
                    return restaurant['name']

        # Chercher dans tous les restaurants
        for _, restaurant in self.restaurants.iterrows():
            if restaurant['name'].lower() in text_lower:
                return restaurant['name']

        return None

    def update_context(self, user_input):
        """Met à jour le contexte avec les entités extraites."""
        city = self.extract_city(user_input)
        if city:
            self.context['ville'] = city

        cuisine = self.extract_cuisine(user_input)
        if cuisine:
            self.context['type_cuisine'] = cuisine

        restaurant = self.extract_restaurant_name(user_input)
        if restaurant:
            self.context['restaurant_courant'] = restaurant

    # ========================================================================
    # RECHERCHE RESTAURANTS
    # ========================================================================

    def search_restaurants(self):
        """Recherche des restaurants selon le contexte."""
        results = self.restaurants.copy()

        if self.context['ville']:
            results = results[
                results['city'].str.lower() == self.context['ville'].lower()
            ]

        if self.context['type_cuisine'] and 'cuisines' in results.columns:
            results = results[
                results['cuisines'].str.lower().str.contains(
                    self.context['type_cuisine'].lower(), na=False
                )
            ]

        self.context['restaurants_filtres'] = results
        return results

    def get_restaurant_by_name(self, name):
        """Récupère un restaurant par son nom."""
        restaurant = self.restaurants[
            self.restaurants['name'].str.lower() == name.lower()
        ]

        if len(restaurant) > 0:
            return restaurant.iloc[0]
        return None

    # ========================================================================
    # GÉNÉRATION DE RÉPONSES
    # ========================================================================

    def response_greeting(self):
        self.context['conversation_started'] = True
        return random.choice(self.responses['greeting'])

    def response_provide_location(self):
        if self.context['ville']:
            return f"Parfait ! {self.context['ville']}. Quel type de cuisine ?"
        return "Je n'ai pas bien saisi la ville. Précisez ?"

    def response_provide_cuisine(self):
        if self.context['type_cuisine']:
            if self.context['ville']:
                return self.response_list_restaurants()
            return f"Cuisine {self.context['type_cuisine']}, bon choix ! Dans quelle ville ?"
        return "Quel type de cuisine préférez-vous ?"

    def response_list_restaurants(self):
        if not self.context['ville']:
            return "Dans quelle ville ?"

        results = self.search_restaurants()

        if len(results) == 0:
            msg = "Désolé, aucun restaurant"
            if self.context['type_cuisine']:
                msg += f" {self.context['type_cuisine']}"
            if self.context['ville']:
                msg += f" à {self.context['ville']}"
            msg += ". Autre recherche ?"
            return msg

        msg = f"Voici les restaurants"
        if self.context['type_cuisine']:
            msg += f" {self.context['type_cuisine']}"
        if self.context['ville']:
            msg += f" à {self.context['ville']}"
        msg += f" ({len(results)} trouvés):\n\n"

        for i, (_, resto) in enumerate(results.head(5).iterrows(), 1):
            msg += f"{i}. 🍽️ **{resto['name']}**\n"
            if 'address' in resto and pd.notna(resto['address']):
                msg += f"   📍 {resto['address']}\n"
            if 'avg_rating' in resto and pd.notna(resto['avg_rating']) and resto['avg_rating'] > 0:
                stars = "⭐" * int(resto['avg_rating'])
                msg += f"   {stars} {resto['avg_rating']:.1f}/2\n"
            if 'price' in resto and pd.notna(resto['price']):
                msg += f"   💰 {resto['price']}\n"
            msg += "\n"

        if len(results) > 5:
            msg += f"... et {len(results)-5} autres.\n"

        msg += "\n💡 Demandez-moi les avis, l'adresse, ou le menu !"

        return msg

    def response_get_reviews(self):
        restaurant_name = self.context['restaurant_courant']

        if not restaurant_name:
            if self.context['restaurants_filtres'] is not None and len(self.context['restaurants_filtres']) > 0:
                restaurant_name = self.context['restaurants_filtres'].iloc[0]['name']
                self.context['restaurant_courant'] = restaurant_name
            else:
                return "Pour quel restaurant voulez-vous les avis ?"

        resto = self.get_restaurant_by_name(restaurant_name)

        if resto is None:
            return f"Je ne trouve pas '{restaurant_name}'."

        msg = f"📊 Avis pour **{resto['name']}**:\n\n"

        if 'avg_rating' in resto and pd.notna(resto['avg_rating']) and resto['avg_rating'] > 0:
            stars = "⭐" * int(resto['avg_rating'])
            msg += f"{stars} **Note:** {resto['avg_rating']:.1f}/2\n"

            if 'nb_reviews' in resto and pd.notna(resto['nb_reviews']):
                msg += f"📝 Basé sur {int(resto['nb_reviews'])} avis\n\n"

            if 'avg_food_rating' in resto and pd.notna(resto['avg_food_rating']):
                msg += f"🍽️ Nourriture: {resto['avg_food_rating']:.1f}/2\n"

            if 'avg_service_rating' in resto and pd.notna(resto['avg_service_rating']):
                msg += f"👨‍🍳 Service: {resto['avg_service_rating']:.1f}/2\n"
        else:
            msg += "Aucun avis disponible.\n"

        return msg

    def response_get_address(self):
        restaurant_name = self.context['restaurant_courant']

        if not restaurant_name:
            if self.context['restaurants_filtres'] is not None and len(self.context['restaurants_filtres']) > 0:
                restaurant_name = self.context['restaurants_filtres'].iloc[0]['name']
                self.context['restaurant_courant'] = restaurant_name
            else:
                return "Pour quel restaurant ?"

        resto = self.get_restaurant_by_name(restaurant_name)

        if resto is None:
            return f"Je ne trouve pas '{restaurant_name}'."

        msg = f"📍 **{resto['name']}**\n\n"

        if 'address' in resto and pd.notna(resto['address']):
            msg += f"Adresse: {resto['address']}\n"
        if 'city' in resto and pd.notna(resto['city']):
            msg += f"Ville: {resto['city']}\n"
        if 'zip' in resto and pd.notna(resto['zip']):
            msg += f"Code postal: {resto['zip']}\n"

        return msg

    def response_get_menu(self):
        restaurant_name = self.context['restaurant_courant']

        if not restaurant_name:
            if self.context['restaurants_filtres'] is not None and len(self.context['restaurants_filtres']) > 0:
                restaurant_name = self.context['restaurants_filtres'].iloc[0]['name']
                self.context['restaurant_courant'] = restaurant_name
            else:
                return "Pour quel restaurant ?"

        resto = self.get_restaurant_by_name(restaurant_name)

        if resto is None:
            return f"Je ne trouve pas '{restaurant_name}'."

        msg = f"🍽️ **{resto['name']}**\n\n"

        if 'cuisines' in resto and pd.notna(resto['cuisines']):
            msg += f"Type: {resto['cuisines']}\n"
        if 'price' in resto and pd.notna(resto['price']):
            msg += f"💰 Prix: {resto['price']}\n"

        msg += "\n💡 Les détails du menu ne sont pas disponibles. Contactez le restaurant."

        return msg

    def response_get_prices(self):
        return self.response_get_menu()

    def response_get_hours(self):
        restaurant_name = self.context['restaurant_courant']
        if not restaurant_name:
            return "Pour quel restaurant ?"
        return f"Les horaires pour {restaurant_name} ne sont pas disponibles. Contactez-les directement."

    def response_get_parking(self):
        restaurant_name = self.context['restaurant_courant']
        if not restaurant_name:
            return "Pour quel restaurant ?"

        resto = self.get_restaurant_by_name(restaurant_name)
        if resto is None:
            return f"Je ne trouve pas '{restaurant_name}'."

        msg = f"🅿️ Parking **{resto['name']}**:\n\n"
        if 'parking_options' in resto and pd.notna(resto['parking_options']):
            msg += f"Options: {resto['parking_options']}\n"
        else:
            msg += "Info parking non disponible.\n"

        return msg

    def response_get_payment(self):
        restaurant_name = self.context['restaurant_courant']
        if not restaurant_name:
            return "Pour quel restaurant ?"

        resto = self.get_restaurant_by_name(restaurant_name)
        if resto is None:
            return f"Je ne trouve pas '{restaurant_name}'."

        msg = f"💳 Paiements **{resto['name']}**:\n\n"
        if 'payment_methods' in resto and pd.notna(resto['payment_methods']):
            msg += f"Acceptés: {resto['payment_methods']}\n"
        else:
            msg += "Info paiement non disponible.\n"

        return msg

    def response_make_reservation(self):
        return "Pour réserver, contactez directement le restaurant. Je peux vous donner l'adresse ?"

    # ========================================================================
    # TRAITEMENT MESSAGE
    # ========================================================================

    def process_message(self, user_input):
        """Traite un message et génère une réponse."""

        # Sauvegarder
        self.history.append({
            'timestamp': datetime.now(),
            'user': user_input,
            'bot': None
        })

        # Extraire entités
        self.update_context(user_input)

        # Prédire intention
        intent, confidence = self.pipeline.predict(user_input, return_confidence=True)

        # Fallback si confiance faible
        if confidence < self.confidence_threshold:
            intent = 'unknown'

        # Générer réponse
        response_map = {
            'greeting': self.response_greeting,
            'provide_location': self.response_provide_location,
            'provide_cuisine': self.response_provide_cuisine,
            'list_restaurants': self.response_list_restaurants,
            'get_reviews': self.response_get_reviews,
            'get_address': self.response_get_address,
            'get_menu': self.response_get_menu,
            'get_prices': self.response_get_prices,
            'get_hours': self.response_get_hours,
            'get_parking': self.response_get_parking,
            'get_payment': self.response_get_payment,
            'make_reservation': self.response_make_reservation,
            'thanks': lambda: random.choice(self.responses['thanks']),
            'goodbye': lambda: random.choice(self.responses['goodbye']),
            'unknown': lambda: random.choice(self.responses['unknown'])
        }

        if intent in response_map:
            response = response_map[intent]()
        else:
            response = random.choice(self.responses['unknown'])

        # Sauvegarder réponse
        self.history[-1]['bot'] = response
        self.history[-1]['intent'] = intent
        self.history[-1]['confidence'] = confidence

        return response, intent, confidence

    def reset_context(self):
        """Réinitialise le contexte."""
        self.context = {
            'ville': None,
            'type_cuisine': None,
            'restaurant_courant': None,
            'restaurants_filtres': None,
            'conversation_started': False
        }


# ============================================================================
# SAUVEGARDE CHATBOT
# ============================================================================

def save_chatbot_class():
    """Sauvegarde la classe RestaurantChatbot dans un fichier."""

    with open('chatbot_class.py', 'w', encoding='utf-8') as f:
        # Écrire tout le code de la classe
        f.write('''#
import pandas as pd
import numpy as np
import random
import pickle
from datetime import datetime

''')

        f.write('# Classe RestaurantChatbot \n')

    print("✅ Classe sauvegardée: chatbot_class.py")


# ============================================================================
# CRÉATION FICHIER STREAMLIT
# ============================================================================

def create_streamlit_app():
    """Crée le fichier app.py pour Streamlit."""

    app_code = '''import streamlit as st
import pandas as pd
import pickle
from datetime import datetime

st.set_page_config(page_title="Assistant Restaurant", page_icon="🍽️", layout="wide")

st.markdown("""
<style>
.user-msg {background-color: #e3f2fd; padding: 1rem; border-radius: 0.5rem; margin: 0.5rem 0;}
.bot-msg {background-color: #fff; padding: 1rem; border-radius: 0.5rem; margin: 0.5rem 0; box-shadow: 0 2px 4px rgba(0,0,0,0.1);}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_chatbot():
    try:
        with open('chatbot_pipeline.pkl', 'rb') as f:
            pipeline = pickle.load(f)
        restaurants = pd.read_csv('restaurants_prepared.csv')

        # Importer RestaurantChatbot
        exec(open('chatbot_class.py').read(), globals())
        chatbot = RestaurantChatbot(restaurants, pipeline)

        return chatbot, restaurants
    except Exception as e:
        st.error(f"Erreur: {e}")
        return None, None

def main():
    st.title("🍽️ Assistant Restaurant")

    chatbot, restaurants = load_chatbot()
    if not chatbot:
        st.stop()

    if 'messages' not in st.session_state:
        st.session_state.messages = []

    with st.sidebar:
        st.header("📊 Stats")
        st.metric("Restaurants", len(restaurants))
        if st.button("🔄 Reset"):
            st.session_state.messages = []
            chatbot.reset_context()
            st.rerun()

    for msg in st.session_state.messages:
        css = "user-msg" if msg['role'] == 'user' else "bot-msg"
        icon = "👤" if msg['role'] == 'user' else "🤖"
        st.markdown(f'<div class="{css}">{icon} {msg["content"]}</div>', unsafe_allow_html=True)

    user_input = st.chat_input("Votre message...")

    if user_input:
        st.session_state.messages.append({'role': 'user', 'content': user_input})
        response, intent, conf = chatbot.process_message(user_input)
        st.session_state.messages.append({'role': 'bot', 'content': response})
        st.rerun()

if __name__ == "__main__":
    main()
'''

    with open('app.py', 'w', encoding='utf-8') as f:
        f.write(app_code)

    print("✅ Fichier Streamlit créé: app.py")


# ============================================================================
# DÉPLOIEMENT AVEC NGROK
# ============================================================================

def deploy_streamlit():
    """Déploie l'app Streamlit avec ngrok."""

    print("\n" + "="*70)
    print("DÉPLOIEMENT STREAMLIT")
    print("="*70 + "\n")

    try:
        from pyngrok import ngrok
        import os

        # Créer l'app
        create_streamlit_app()

        # Sauvegarder classe
        save_chatbot_class()

        # Tuer tunnels existants
        ngrok.kill()

        # Créer tunnel
        public_url = ngrok.connect(8501)
        print(f"\n🌐 Application accessible à:\n{public_url}\n")

        # Lancer Streamlit
        print("🚀 Lancement de Streamlit...\n")
        os.system("streamlit run app.py --server.port 8501 --server.headless true &")

        print("✅ Streamlit lancé !")
        print("📱 Ouvrez l'URL ci-dessus dans votre navigateur\n")

    except Exception as e:
        print(f"\n❌ Erreur déploiement: {e}")
        import traceback
        traceback.print_exc()


# ============================================================================
# EXÉCUTION
# ============================================================================

# Charger les données
print("\n📂 Chargement des données...")
restaurants = pd.read_csv('restaurants_prepared.csv')
with open('chatbot_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

# Créer chatbot
print("🤖 Création du chatbot...")
chatbot = RestaurantChatbot(restaurants, pipeline, confidence_threshold=0.70)

print("\n✅ Chatbot créé avec succès !")

# Test rapide
print("\n" + "="*70)
print("TEST RAPIDE DU CHATBOT")
print("="*70 + "\n")

test_messages = [
    "Bonjour",
    "Je suis à San Luis Potosi",
    "Je veux du mexicain",
    "Liste les restaurants"
]

for msg in test_messages:
    response, intent, conf = chatbot.process_message(msg)
    print(f"👤 Vous: {msg}")
    print(f"🤖 Bot: {response[:100]}...")
    print(f"   [Intent: {intent}, Confiance: {conf:.2f}]\n")

# Déployer Streamlit
print("\n" + "="*70)
print("VOULEZ-VOUS DÉPLOYER L'INTERFACE STREAMLIT ?")
print("="*70 + "\n")

print("Pour déployer, exécutez dans une nouvelle cellule:")
print(">>> deploy_streamlit()")

print("\n✅ PROJET COMPLET TERMINÉ !")
print("="*70 + "\n")


📂 Chargement des données...
🤖 Création du chatbot...

✅ Chatbot créé avec succès !

TEST RAPIDE DU CHATBOT

👤 Vous: Bonjour
🤖 Bot: Bonjour ! 🍽️ Je suis votre assistant restaurant. Dans quelle ville cherchez-vous ?...
   [Intent: greeting, Confiance: 0.98]

👤 Vous: Je suis à San Luis Potosi
🤖 Bot: Parfait ! San Luis Potosi. Quel type de cuisine ?...
   [Intent: provide_location, Confiance: 0.79]

👤 Vous: Je veux du mexicain
🤖 Bot: Voici les restaurants Mexican à San Luis Potosi (9 trouvés):

1. 🍽️ **El Rinc�n de San Francisco**
 ...
   [Intent: provide_cuisine, Confiance: 0.84]

👤 Vous: Liste les restaurants
🤖 Bot: Voici les restaurants Mexican à San Luis Potosi (9 trouvés):

1. 🍽️ **El Rinc�n de San Francisco**
 ...
   [Intent: list_restaurants, Confiance: 0.94]


VOULEZ-VOUS DÉPLOYER L'INTERFACE STREAMLIT ?

Pour déployer, exécutez dans une nouvelle cellule:
>>> deploy_streamlit()

✅ PROJET COMPLET TERMINÉ !



In [43]:
# Dans une nouvelle cellule
deploy_streamlit()


DÉPLOIEMENT STREAMLIT

✅ Fichier Streamlit créé: app.py
✅ Classe sauvegardée: chatbot_class.py

🌐 Application accessible à:
NgrokTunnel: "https://<url-ngrok>.ngrok-free.dev" -> "http://localhost:8501"

🚀 Lancement de Streamlit...

✅ Streamlit lancé !
📱 Ouvrez l'URL ci-dessus dans votre navigateur



In [44]:
from pyngrok import ngrok

# Configuration du token ngrok : à définir dans la variable d'environnement NGROK_AUTHTOKEN
import os
ngrok.set_auth_token(os.environ.get("NGROK_AUTHTOKEN", "VOTRE_TOKEN_NGROK"))  # ne jamais publier son token

print("✅ ngrok est maintenant configuré avec ton token !")

✅ ngrok est maintenant configuré avec ton token !


In [45]:
def create_streamlit_app():
    app_code = r'''import streamlit as st
import pandas as pd
import pickle
import re
import string
import nltk
import numpy as np
import random
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# --- CLASSES NLP (Nécessaires pour Pickle) ---
class TextPreprocessor:
    def __init__(self, language='french'):
        self.language = language
        self.stop_words = set(stopwords.words(language))
        self.lemmatizer = WordNetLemmatizer()
    def clean_text(self, text):
        text = text.lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        return re.sub(r"\s+", " ", text).strip()
    def preprocess(self, text):
        text = self.clean_text(text)
        tokens = word_tokenize(text, language=self.language)
        tokens = [t for t in tokens if t not in self.stop_words]
        return " ".join([self.lemmatizer.lemmatize(t) for t in tokens])

class IntentClassifier:
    def __init__(self):
        self.best_model = None
    def predict(self, X): return self.best_model.predict(X)
    def predict_proba(self, X): return self.best_model.predict_proba(X)

class NLPPipeline:
    def __init__(self, vectorization_method='tfidf', ngram_range=(1, 2)):
        self.preprocessor = TextPreprocessor()
        self.vectorizer = None
        self.classifier = IntentClassifier()
    def predict(self, text, return_confidence=True):
        processed_text = self.preprocessor.preprocess(text)
        X = self.vectorizer.transform([processed_text])
        intent = self.classifier.predict(X)[0]
        probas = self.classifier.predict_proba(X)
        confidence = np.max(probas[0])
        return intent, confidence

class RestaurantChatbot:
    def __init__(self, restaurants_df, nlp_pipeline):
        self.restaurants = restaurants_df
        self.pipeline = nlp_pipeline
        self.context = {'ville': None}
    def process_message(self, user_input):
        intent, conf = self.pipeline.predict(user_input)
        if conf < 0.60: return "Désolé, je ne comprends pas votre demande.", "unknown"
        if intent == 'greeting': return "Bonjour ! Dans quelle ville cherchez-vous un restaurant ?", intent
        return f"J'ai détecté l'intention : {intent} (Confiance: {conf:.2f})", intent

# --- INTERFACE ---
@st.cache_resource
def load_data():
    with open('chatbot_pipeline.pkl', 'rb') as f:
        pipeline = pickle.load(f)
    restaurants = pd.read_csv('restaurants_prepared.csv')
    return RestaurantChatbot(restaurants, pipeline)

def main():
    st.title("🍽️ Assistant Restaurant")
    try:
        bot = load_data()
        if 'msgs' not in st.session_state: st.session_state.msgs = []
        for m in st.session_state.msgs:
            with st.chat_message(m["role"]): st.markdown(m["content"])

        user_input = st.chat_input("Posez votre question...")
        if user_input:
            st.session_state.msgs.append({"role": "user", "content": user_input})
            with st.chat_message("user"): st.markdown(user_input)
            ans, _ = bot.process_message(user_input)
            st.session_state.msgs.append({"role": "assistant", "content": ans})
            with st.chat_message("assistant"): st.markdown(ans)
    except Exception as e:
        st.error(f"Erreur : {e}")

if __name__ == "__main__":
    main()
'''
    with open('app.py', 'w', encoding='utf-8') as f:
        f.write(app_code)
    print("✅ app.py généré sans erreur de syntaxe.")

In [46]:
deploy_streamlit()



DÉPLOIEMENT STREAMLIT

✅ app.py généré sans erreur de syntaxe.
✅ Classe sauvegardée: chatbot_class.py

🌐 Application accessible à:
NgrokTunnel: "https://<url-ngrok>.ngrok-free.dev" -> "http://localhost:8501"

🚀 Lancement de Streamlit...

✅ Streamlit lancé !
📱 Ouvrez l'URL ci-dessus dans votre navigateur

